# PublicHearingBR — Separação de políticos por estado (UF)

Pré-processamento referente à issue **#4 "Fazer Script de pré-processamento de dados - Separar por estado"**
(sub-issues **#11 "Retirar os jornalistas"** e **#12 "Gerariam dois conjuntos de dados"**).

Dataset: [`unicamp-dl/PublicHearingBR`](https://huggingface.co/datasets/unicamp-dl/PublicHearingBR) — transcrições de audiências públicas da Câmara dos Deputados, em 2 arquivos:

- **`PublicHearingBR_LDS.jsonl`** (206 registros): `id`, `transcricao` (texto completo da audiência), `materia` (notícia), `metadados.envolvidos` (lista de pessoas citadas na notícia, cada uma com `nome`, `cargo`, `opinioes`).
- **`PublicHearingBR_NLI.jsonl`** (4238 registros): `id`, `metadados_extraidos.envolvidos` — estrutura parecida, mas **sem** o campo `transcricao`.

## Premissas assumidas (confirmadas com o Leonardo no chat)

1. Os **dois arquivos** são processados, cada um gerando seu próprio conjunto de saídas separadas por estado — é isso que dá os "dois conjuntos de dados" da #12.
2. Jornalistas são removidos de **todos** os `envolvidos` antes de separar por estado (#11), restando só políticos.
3. Cada registro (linha) original é mantido **inteiro** (mesma estrutura), só a lista `envolvidos` é filtrada. Um registro entra no arquivo de um estado se, depois de remover jornalistas, sobrar pelo menos um envolvido daquele estado. **Um mesmo registro pode cair em mais de um arquivo de estado** se citar políticos de estados diferentes — isso preserva a informação em vez de forçar uma escolha arbitrária de "estado principal".
4. **Extração do estado (UF):** o dataset não tem um campo `estado` pronto. Duas fontes são usadas, em ordem de prioridade:
   - **Tag do orador na transcrição** (só existe no LDS): o padrão confirmado no preview público do dataset é `(Nome. Partido - UF)`, ex. `PRESIDENTE(Lucas Redecker. Bloco/PSDB - RS)`. O notebook também tenta uma segunda forma comum em transcrições da Câmara, `NOME (Partido - UF)` (nome fora do parênteses), como fallback — essa segunda forma **não** foi confirmada nos dados reais, só é uma rede de segurança.
   - **Campo `cargo`** (LDS e NLI): tentamos extrair um padrão tipo `... - UF` ou `.../UF` diretamente do texto do cargo (esse é o único caminho possível pro NLI, que não tem transcrição).
5. **⚠️ Limitação importante:** não tive acesso de execução ao dataset completo neste ambiente (rede restrita), então o parsing foi desenhado a partir dos exemplos visíveis no preview público do Hugging Face (que mostrou o padrão `(Nome. Partido - UF)` em 8/8 transcrições, mas nenhum exemplo de `cargo` de político — só de jornalista). **Rode a seção de diagnóstico abaixo antes de confiar no resto do pipeline** e ajuste `JOURNALIST_KEYWORDS` / os regex de UF se os padrões reais dos dados forem diferentes.


## 1. Setup e download dos dados

In [ ]:
# Instala dependências (idempotente — pode rodar de novo sem problema)
%pip install -q requests


In [ ]:
import json
import re
import copy
import random
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter

import requests

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Padrão de tag de orador nas transcrições da Câmara: "(Nome. Partido - UF)".
# Definido aqui (e não só na seção de diagnóstico) porque a seção 4 depende dele
# mesmo que a seção 2 seja pulada.
SPEAKER_TAG_PATTERN = re.compile(r"\(([^()]{3,80})\)")

HF_BASE = "https://huggingface.co/datasets/unicamp-dl/PublicHearingBR/resolve/main"
FILES = {
    "lds": "PublicHearingBR_LDS.jsonl",
    "nli": "PublicHearingBR_NLI.jsonl",
}


def download_file(filename: str, dest_dir: Path = DATA_DIR) -> Path:
    dest = dest_dir / filename
    if dest.exists():
        print(f"[ok] {filename} já baixado ({dest.stat().st_size / 1e6:.1f} MB)")
        return dest
    url = f"{HF_BASE}/{filename}?download=true"
    print(f"Baixando {filename} de {url} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"[ok] {filename} salvo em {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


lds_path = download_file(FILES["lds"])
nli_path = download_file(FILES["nli"])


In [ ]:
def load_jsonl(path: Path):
    registros = []
    with open(path, encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                registros.append(json.loads(linha))
    return registros


phbr_lds = load_jsonl(lds_path)
phbr_nli = load_jsonl(nli_path)

print(f"PublicHearingBR_LDS: {len(phbr_lds)} registros")
print(f"PublicHearingBR_NLI: {len(phbr_nli)} registros")


## 2. Diagnóstico — rode isto antes de confiar no resto

Aqui a gente olha uma amostra real de `cargo` (pra calibrar a lista de jornalistas e o regex de UF)
e uma amostra das tags de orador extraídas da `transcricao`. Se os padrões abaixo não baterem com o
que você está vendo, ajuste as células da seção 3 e 4 antes de gerar os arquivos finais.

In [ ]:
def coletar_cargos(dataset, chave_envolvidos):
    cargos = []
    node_key, sub_key = chave_envolvidos
    for r in dataset:
        for env in r.get(node_key, {}).get(sub_key, []):
            cargos.append(env.get("cargo", ""))
    return cargos


cargos_lds = coletar_cargos(phbr_lds, ("metadados", "envolvidos"))
cargos_nli = coletar_cargos(phbr_nli, ("metadados_extraidos", "envolvidos"))

print(f"Total de envolvidos (LDS): {len(cargos_lds)}")
print(f"Total de envolvidos (NLI): {len(cargos_nli)}")

print("\nAmostra de 'cargo' no LDS:")
for c in random.sample(cargos_lds, min(20, len(cargos_lds))):
    print(" -", c)

print("\nAmostra de 'cargo' no NLI:")
for c in random.sample(cargos_nli, min(20, len(cargos_nli))):
    print(" -", c)


In [ ]:
# Amostra das tags de orador extraídas da transcrição (só existe no LDS)
# (SPEAKER_TAG_PATTERN já foi definido na célula de setup, seção 1)
amostra_tags = []
for r in random.sample(phbr_lds, min(5, len(phbr_lds))):
    tags = SPEAKER_TAG_PATTERN.findall(r.get("transcricao", ""))
    amostra_tags.extend(tags[:10])

print("Amostra de conteúdo entre parênteses na transcrição (procurando o padrão 'Nome. Partido - UF'):")
for t in amostra_tags[:40]:
    print(" -", t)


## 3. Remoção de jornalistas

In [ ]:
def normalize(texto: str) -> str:
    if not texto:
        return ""
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()


# Lista configurável — ajuste depois de ver a amostra de 'cargo' da seção 2
JOURNALIST_KEYWORDS = [
    "jornalista", "correspondente", "reporter", "apresentador", "apresentadora",
    "colunista", "comentarista", "blogueiro", "blogueira", "redator", "redatora",
    "ancora", "editor-chefe", "editora-chefe", "editor de jornal", "editora de jornal",
    "imprensa",
]


def is_journalist(cargo: str) -> bool:
    cargo_norm = normalize(cargo)
    return any(kw in cargo_norm for kw in JOURNALIST_KEYWORDS)


## 4. Extração do estado (UF)

In [ ]:
UF_LIST = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS",
    "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC",
    "SP", "SE", "TO",
]
UF_SET = set(UF_LIST)

# Ex.: "PSDB - RS", "Bloco/UNIÃO - MG", ".../SP"
CARGO_UF_PATTERN = re.compile(r"(?:-\s*|/)\s*([A-Z]{2})\b")


def extract_uf_from_cargo(cargo: str):
    if not cargo:
        return None
    for m in CARGO_UF_PATTERN.finditer(cargo):
        if m.group(1) in UF_SET:
            return m.group(1)
    return None


# Fallback pra um 2o formato comum em transcricoes da Camara, em que o nome
# do orador vem em maiúsculas ANTES do parênteses, e só "Partido - UF" fica
# dentro (ex.: "O SR. FULANO DE TAL (PT - SP) - ..."). Só a forma "(Nome.
# Partido - UF)" (tudo dentro do parênteses) foi confirmada no preview público
# do dataset — essa segunda forma é defensiva; confira na seção 2 se ela é
# necessária nos seus dados.
NAME_BEFORE_PAREN_PATTERN = re.compile(
    r"([A-ZÀ-ÜÇ][A-ZÀ-ÜÇ\.\s]{2,60}?)\s*\(([^()]{1,40}-\s*[A-Z]{2})\)"
)


def build_speaker_uf_map(transcricao: str) -> dict:
    """Varre a transcrição em busca de tags de orador e devolve
    {nome_normalizado: uf}. Tenta duas formas (ver comentário acima)."""
    mapping = {}
    if not transcricao:
        return mapping

    # Forma A (confirmada): "(Nome. Partido - UF)" tudo dentro do parênteses
    for content in SPEAKER_TAG_PATTERN.findall(transcricao):
        if "." not in content:
            continue
        nome_part, _, resto = content.partition(".")
        nome = nome_part.strip()
        m = re.search(r"-\s*([A-Z]{2})\s*$", resto.strip())
        if nome and m and m.group(1) in UF_SET:
            mapping[normalize(nome)] = m.group(1)

    # Forma B (fallback): nome antes do parênteses, só "Partido - UF" dentro
    for m in NAME_BEFORE_PAREN_PATTERN.finditer(transcricao):
        nome, resto = m.group(1).strip(" ."), m.group(2)
        m_uf = re.search(r"-\s*([A-Z]{2})\s*$", resto.strip())
        if nome and m_uf and m_uf.group(1) in UF_SET:
            mapping.setdefault(normalize(nome), m_uf.group(1))

    return mapping


def match_name_to_uf(nome: str, speaker_map: dict):
    nome_norm = normalize(nome)
    if not nome_norm or not speaker_map:
        return None
    if nome_norm in speaker_map:
        return speaker_map[nome_norm]
    nome_tokens = set(nome_norm.split())
    for speaker_name, uf in speaker_map.items():
        if nome_norm in speaker_name or speaker_name in nome_norm:
            return uf
        # exige >=2 tokens em comum pra evitar falso-positivo por sobrenome comum
        if len(nome_tokens & set(speaker_name.split())) >= 2:
            return uf
    return None


def resolve_uf(envolvido: dict, speaker_map: dict):
    uf = extract_uf_from_cargo(envolvido.get("cargo", ""))
    if uf:
        return uf
    return match_name_to_uf(envolvido.get("nome", ""), speaker_map)


## 5. Separar por estado e salvar

In [ ]:
def process_record(record: dict, envolvidos_path: tuple, usa_transcricao: bool):
    """Devolve (registro_com_envolvidos_filtrados, ufs_encontradas, n_jornalistas_removidos).
    A estrutura original do registro é preservada — só a lista de envolvidos é filtrada."""
    novo = copy.deepcopy(record)
    node_key, sub_key = envolvidos_path
    envolvidos = novo.get(node_key, {}).get(sub_key, [])

    speaker_map = build_speaker_uf_map(record.get("transcricao", "")) if usa_transcricao else {}

    mantidos, ufs_encontradas, n_jornalistas = [], set(), 0
    for env in envolvidos:
        if is_journalist(env.get("cargo", "")):
            n_jornalistas += 1
            continue
        uf = resolve_uf(env, speaker_map)
        if uf:
            ufs_encontradas.add(uf)
        mantidos.append(env)

    novo[node_key][sub_key] = mantidos
    return novo, ufs_encontradas, n_jornalistas


def separar_por_estado(dataset: list, envolvidos_path: tuple, usa_transcricao: bool, output_dir: Path):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    por_estado = defaultdict(list)
    sem_estado = []
    total_jornalistas = 0

    for record in dataset:
        novo, ufs, n_jorn = process_record(record, envolvidos_path, usa_transcricao)
        total_jornalistas += n_jorn
        if ufs:
            for uf in ufs:
                por_estado[uf].append(novo)
        else:
            sem_estado.append(novo)

    for uf, registros in sorted(por_estado.items()):
        with open(output_dir / f"{uf}.jsonl", "w", encoding="utf-8") as f:
            for r in registros:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    if sem_estado:
        with open(output_dir / "SEM_ESTADO.jsonl", "w", encoding="utf-8") as f:
            for r in sem_estado:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    resumo = {uf: len(regs) for uf, regs in sorted(por_estado.items())}
    resumo["SEM_ESTADO"] = len(sem_estado)
    return resumo, total_jornalistas


In [ ]:
resumo_lds, jornalistas_removidos_lds = separar_por_estado(
    phbr_lds,
    envolvidos_path=("metadados", "envolvidos"),
    usa_transcricao=True,
    output_dir=OUTPUT_DIR / "PublicHearingBR_LDS_por_estado",
)

resumo_nli, jornalistas_removidos_nli = separar_por_estado(
    phbr_nli,
    envolvidos_path=("metadados_extraidos", "envolvidos"),
    usa_transcricao=False,
    output_dir=OUTPUT_DIR / "PublicHearingBR_NLI_por_estado",
)


## 6. Resumo final

In [ ]:
def print_resumo(nome_dataset, resumo, jornalistas_removidos, total_original):
    print(f"=== {nome_dataset} ===")
    print(f"Registros originais: {total_original}")
    print(f"Envolvidos removidos por serem jornalistas: {jornalistas_removidos}")
    print(f"Registros sem nenhum estado identificado: {resumo.get('SEM_ESTADO', 0)}")
    print("Registros por estado (um registro pode contar em mais de um estado):")
    for uf, n in sorted(resumo.items()):
        if uf != "SEM_ESTADO":
            print(f"  {uf}: {n}")
    print()


print_resumo("PublicHearingBR_LDS", resumo_lds, jornalistas_removidos_lds, len(phbr_lds))
print_resumo("PublicHearingBR_NLI", resumo_nli, jornalistas_removidos_nli, len(phbr_nli))

print(f"Arquivos gerados em: {(OUTPUT_DIR / 'PublicHearingBR_LDS_por_estado').resolve()}")
print(f"Arquivos gerados em: {(OUTPUT_DIR / 'PublicHearingBR_NLI_por_estado').resolve()}")
